In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.gold;

In [0]:
from pyspark.sql.functions import col

In [0]:
races_df = spark.read.table("f1.silver.races")
results_df = spark.read.table("f1.silver.results")
drivers_df = spark.read.table("f1.silver.drivers")
constructors_df = spark.read.table("f1.silver.constructors")

In [0]:
races_selected_df = races_df.select(
    "race_id",
    "race_name",
    "year",
    "round",
    "circuit_id",
    "race_timestamp"
)

results_selected_df = results_df.select(
    "result_id",
    "race_id",
    "driver_id",
    "constructor_id",
    "grid",
    "position",
    "position_text",
    "position_order",
    "points",
    "laps",
    "time",
    "milliseconds",
    "fastest_lap",
    "fastest_lap_time",
    "fastest_lap_time_ms",
    "fastest_lap_speed",
    "status_id"
)

drivers_selected_df = drivers_df.select(
    "driver_id",
    "driver_ref",
    "name",
    "nationality",
    "code"
)

constructors_selected_df = constructors_df.select(
    "constructor_id",
    "constructor_ref",
    "constructor_name",
    "nationality"
)


In [0]:
display(results_selected_df)

In [0]:
race_results_df = results_selected_df \
    .join(
        races_selected_df,
        results_selected_df.race_id == races_selected_df.race_id,
        "inner"
    ) \
    .join(
        drivers_selected_df,
        results_selected_df.driver_id == drivers_selected_df.driver_id,
        "inner"
    ) \
    .join(
        constructors_selected_df,
        results_selected_df.constructor_id == constructors_selected_df.constructor_id,
        "inner"
    )

In [0]:
race_results_final_df = race_results_df.select(

    results_selected_df.result_id,
    
    races_selected_df.race_id,
    races_selected_df.race_name,
    races_selected_df.year,
    races_selected_df.round,
    races_selected_df.circuit_id,
    races_selected_df.race_timestamp,

    drivers_selected_df.driver_id,
    drivers_selected_df.name.alias("driver_name"),
    drivers_selected_df.driver_ref,
    drivers_selected_df.nationality.alias("driver_nationality"),
    drivers_selected_df.code.alias("driver_code"),

    constructors_selected_df.constructor_id,
    constructors_selected_df.constructor_name,
    constructors_selected_df.constructor_ref,
    constructors_selected_df.nationality.alias("constructor_nationality"),

    results_selected_df.grid,
    results_selected_df.position,
    results_selected_df.position_text,
    results_selected_df.position_order,
    results_selected_df.points,
    results_selected_df.laps,
    results_selected_df.time,
    results_selected_df.milliseconds,
    results_selected_df.fastest_lap,
    results_selected_df.fastest_lap_time,
    results_selected_df.fastest_lap_time_ms,
    results_selected_df.fastest_lap_speed,
    results_selected_df.status_id
)

In [0]:
race_results_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("f1.gold.race_results")

In [0]:
%sql

SELECT *
FROM f1.gold.race_results
WHERE race_id = 18 AND year = '2008'